# GMCC — Global Mean-Centered Correlation

A walkthrough of the similarity measure introduced in:

> Wiyono, S., Adji, T. B., Nugroho, H. A., & Wibirama, S. (2026).
> *Global mean-centered correlation: A variance-reduced and theoretically grounded similarity measure for sparse data environments.*
> **Information Sciences, 749**, 123502. [doi:10.1016/j.ins.2026.123502](https://doi.org/10.1016/j.ins.2026.123502)

GMCC replaces the entity-specific **local** mean centering of Pearson Correlation with a single population-level **global** mean $\mu$:

$$\text{GMCC}(\mathbf{x}, \mathbf{y}) = \frac{\sum_{i=1}^{n} (x_i - \mu)(y_i - \mu)}{\sqrt{\sum_{i=1}^{n} (x_i - \mu)^2}\;\sqrt{\sum_{i=1}^{n} (y_i - \mu)^2}}
\qquad\text{where}\qquad
\mu = \frac{1}{N}\sum_{k=1}^{N} z_k$$

The global mean is computed **once** over all $N$ observed values in the dataset and reused for every pair. There is no hyperparameter to tune.

### What kind of data does this apply to?

**Any** dataset of entities characterised by partially observed, real-valued attributes. Following Section 3.1 of the paper, each observed value is "a measured quantity such as a rating, score, or feature value" — rows could be users and items, documents and terms, sensors and time points, samples and genes, annotators and judged items.

Nothing in the method assumes a rating scale, a bounded range, integer values, or a recommender-system setting. Sections 1–4 below use ratings on a 1–5 scale **only because they are familiar and easy to read by hand**; section 5 runs the same comparisons on sensor measurements in °C to show the scale is not assumed.

*This notebook needs `pandas` in addition to `numpy`, purely for table formatting.*

In [1]:
import numpy as np
import pandas as pd

from gmcc import global_mean, gmcc, gmcc_from_matrix, gmcc_similarity_matrix
from baselines import cosine, pearson

pd.set_option("display.precision", 3)

## 1. The preprocessing step: computing $\mu$

Four users rate three items on a 1–5 scale. Two of them give flat ratings, two give rising ones.

In [2]:
ratings = np.array([
    [1.0, 1.0, 1.0],   # U1: consistently low ratings
    [5.0, 5.0, 5.0],   # U2: consistently high ratings
    [1.0, 2.0, 3.0],   # U3: rising, below the population baseline
    [3.0, 4.0, 5.0],   # U4: rising, at or above the population baseline
])

pd.DataFrame(ratings,
             index=["U1", "U2", "U3", "U4"],
             columns=["Item A", "Item B", "Item C"]).astype(int)

,Item A,Item B,Item C
U1,1,1,1
U2,5,5,5
U3,1,2,3
U4,3,4,5


In [3]:
mu = global_mean(ratings)
print(f"Global mean  mu = {mu}")
print(f"(sum of all {ratings.size} observed values = {ratings.sum():.0f};  {ratings.sum():.0f} / {ratings.size} = {mu})")

Global mean  mu = 3.0
(sum of all 12 observed values = 36;  36 / 12 = 3.0)


Compute this **once**. It is a property of the dataset, not of any particular pair — recomputing it per pair would discard the variance reduction that motivates the measure.

## 2. How the three measures differ

Each row below is a pair of rating vectors. Watch where Cosine and Pearson break down.

In [4]:
cases = [
    ("U1 vs U2",  [1, 1, 1], [5, 5, 5], "constant profiles on opposite sides of mu"),
    ("U3 vs U4",  [1, 2, 3], [3, 4, 5], "same rising pattern, different level"),
    ("U1 vs U3",  [1, 1, 1], [1, 2, 3], "both below the population baseline"),
    ("U2 vs U2",  [5, 5, 5], [5, 5, 5], "identical constant profiles"),
    ("zero row",  [3, 4, 5], [0, 0, 0], "all-zero but genuinely observed profile"),
]

rows = []
for label, x, y, note in cases:
    rows.append({
        "pair": label,
        "x": str(x),
        "y": str(y),
        "Cosine": cosine(x, y),
        "Pearson": pearson(x, y),
        "GMCC": gmcc(x, y, mu),
        "what is happening": note,
    })

pd.DataFrame(rows).set_index("pair").fillna("undefined")

,x,y,Cosine,Pearson,GMCC,what is happening
pair,,,,,,
U1 vs U2,"[1, 1, 1]","[5, 5, 5]",1.0,undefined,-1.000,constant profiles on opposite sides of mu
U3 vs U4,"[1, 2, 3]","[3, 4, 5]",0.983,1.0,-0.200,"same rising pattern, different level"
U1 vs U3,"[1, 1, 1]","[1, 2, 3]",0.926,undefined,0.775,both below the population baseline
U2 vs U2,"[5, 5, 5]","[5, 5, 5]",1.0,undefined,1.000,identical constant profiles
zero row,"[3, 4, 5]","[0, 0, 0]",undefined,undefined,-0.775,all-zero but genuinely observed profile


Reading the table:

| Pair | What goes wrong, and what GMCC does |
|---|---|
| **U1 vs U2** | Cosine returns `+1.000` — the two vectors point in the same direction, so it calls them identical. But U1 rates everything low and U2 rates everything high: they are opposites. Pearson cannot judge at all (both have zero variance). GMCC returns `−1.000`. |
| **U3 vs U4** | Pearson returns `+1.000` from the shared rising pattern. GMCC returns `−0.200`, because U3 sits below the population baseline while U4 sits at or above it. **The two measures disagree in sign.** |
| **U1 vs U3** | Pearson is undefined (U1 is constant). GMCC returns `+0.775`: both users sit below $\mu$. |
| **zero row** | An all-zero vector breaks Cosine's normalization — division by a zero norm. GMCC is unaffected. |

## 3. When is GMCC undefined?

Only in one degenerate case: when *every* value of an entity equals $\mu$ exactly, so its centered vector collapses to the origin.

A constant-valued profile whose value differs from $\mu$ stays well defined and evaluates to $\text{sign}\big((c_x - \mu)(c_y - \mu)\big)$ — **Proposition 3** in the paper. This is precisely the case where Pearson always fails.

In [5]:
rows = []
for c_x, c_y in [(1, 5), (5, 1), (1, 2), (4, 5), (3, 2)]:
    x, y = [c_x] * 4, [c_y] * 4
    rows.append({
        "c_x": c_x, "c_y": c_y,
        "c_x vs mu": "below" if c_x < mu else ("above" if c_x > mu else "EQUAL"),
        "c_y vs mu": "below" if c_y < mu else ("above" if c_y > mu else "EQUAL"),
        "Pearson": pearson(x, y),
        "GMCC": gmcc(x, y, mu),
    })

pd.DataFrame(rows).fillna("undefined")

,c_x,c_y,c_x vs mu,c_y vs mu,Pearson,GMCC
0,1,5,below,above,undefined,-1.0
1,5,1,above,below,undefined,-1.0
2,1,2,below,below,undefined,1.0
3,4,5,above,above,undefined,1.0
4,3,2,EQUAL,below,undefined,undefined


The last row is the single undefined case: $c_x = \mu$, so GMCC has no direction to report either. Everywhere else GMCC gives an answer while Pearson gives none.

## 4. Sparse data with missing entries

Mark unobserved entries with `NaN`. Only attributes observed by **both** entities enter the computation — the co-observation set $I_{uv}$ of the paper.

In [6]:
sparse = np.array([
    [5.0,    3.0,    np.nan, 1.0],
    [4.0,    np.nan, 2.0,    1.0],
    [np.nan, 5.0,    4.0,    np.nan],
    [1.0,    1.0,    2.0,    5.0],
])

users = ["U1", "U2", "U3", "U4"]
pd.DataFrame(sparse, index=users, columns=["I1", "I2", "I3", "I4"])

,I1,I2,I3,I4
U1,5.0,3.0,NaN,1.0
U2,4.0,NaN,2.0,1.0
U3,NaN,5.0,4.0,NaN
U4,1.0,1.0,2.0,5.0


In [7]:
mu_sparse = global_mean(sparse)        # NaN entries are ignored
n_observed = int(np.sum(~np.isnan(sparse)))
print(f"{n_observed} observed values out of {sparse.size} cells "
      f"({100 * n_observed / sparse.size:.0f}% dense)")
print(f"mu = {mu_sparse:.4f}")

12 observed values out of 16 cells (75% dense)
mu = 2.8333


In [8]:
# min_co_observations guards against pairs that share too little data
S = gmcc_similarity_matrix(sparse, mu=mu_sparse, min_co_observations=2)
pd.DataFrame(S, index=users, columns=users).fillna("--")

,U1,U2,U3,U4
U1,1.0,0.955,--,-0.859
U2,0.955,1.0,--,-0.787
U3,--,--,1.0,-0.998
U4,-0.859,-0.787,-0.998,1.000


U3 shares only one observed item with U1 and with U2, so those pairs fall below the `min_co_observations=2` threshold and are returned as `nan` rather than computed from a single point.

## 5. The scale is not assumed: a non-rating domain

Four weather stations report temperature in °C at three time points. No ratings, no 1–5 scale, no recommender system — and $\mu$ is simply whatever the data says it is.

In [9]:
readings = np.array([
    [21.4, 22.1, 23.0],   # ST1: consistently cooler than the network
    [28.2, 29.0, 30.1],   # ST2: consistently warmer than the network
    [21.0, 25.3, 29.4],   # ST3: swings across the network mean
    [25.2, 25.2, 25.2],   # ST4: flat, close to the network mean
])
stations = ["ST1", "ST2", "ST3", "ST4"]

mu_temp = global_mean(readings)
print(f"mu = {mu_temp:.3f} degrees C  (derived from the data, not assumed)")

pd.DataFrame(readings, index=stations, columns=["t1", "t2", "t3"])

mu = 25.425 degrees C  (derived from the data, not assumed)


,t1,t2,t3
ST1,21.4,22.1,23.0
ST2,28.2,29.0,30.1
ST3,21.0,25.3,29.4
ST4,25.2,25.2,25.2


In [10]:
rows = []
for a, b, note in [
    (0, 1, "both baselines are misled by the shared upward trend"),
    (0, 3, "ST4 is flat, so Pearson has no variance to work with"),
    (2, 3, "ST3 straddles the mean that ST4 sits on"),
]:
    rows.append({
        "pair": f"{stations[a]} vs {stations[b]}",
        "Cosine": cosine(readings[a], readings[b]),
        "Pearson": pearson(readings[a], readings[b]),
        "GMCC": gmcc(readings[a], readings[b], mu_temp),
        "what is happening": note,
    })

pd.DataFrame(rows).set_index("pair").fillna("undefined")

,Cosine,Pearson,GMCC,what is happening
pair,,,,
ST1 vs ST2,1.000,1.0,-0.918,both baselines are misled by the shared upward...
ST1 vs ST4,1.000,undefined,0.980,"ST4 is flat, so Pearson has no variance to wor..."
ST3 vs ST4,0.991,undefined,0.056,ST3 straddles the mean that ST4 sits on


Look at the first row: **Cosine and Pearson both report `+1.000`** — they agree, and they are both misled. ST1 reads below the network mean at every time point while ST2 reads above it at every time point. GMCC reports `−0.918`.

Same behaviour as the rating examples, on an entirely different scale, because $\mu$ is fitted to the data rather than fixed by the domain.

## 6. Train/test discipline

In a predictive setting, fit $\mu$ on the **training split only**. Computing it over the full dataset lets test information leak into the similarity computation.

In [11]:
train = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 3.0]])
test  = np.array([[5.0, 5.0, 5.0], [4.0, 5.0, 4.0]])

mu_train = global_mean(train)
mu_all   = global_mean(np.vstack([train, test]))

print(f"mu from training split only : {mu_train:.4f}   <- use this")
print(f"mu from train + test        : {mu_all:.4f}   <- leaks test data")

mu from training split only : 3.0000   <- use this
mu from train + test        : 3.8333   <- leaks test data


```python
mu = global_mean(train_matrix)                   # fit on training data
S  = gmcc_similarity_matrix(full_matrix, mu=mu)  # then apply
```

For reference, the paper's MovieLens 100K experiment (Section 5.1) uses an 80/20 user-based temporal split and reports $\mu = 3.5297$ computed on the training set alone.

## 7. Properties worth knowing

The measure is bounded, symmetric, and invariant to a shift or positive scaling applied to the values and $\mu$ together.

In [12]:
rng = np.random.default_rng(20260421)
x = rng.uniform(1, 5, size=8)
y = rng.uniform(1, 5, size=8)

base = gmcc(x, y, mu)
print(f"GMCC(x, y)               = {base:+.6f}")
print(f"GMCC(y, x)               = {gmcc(y, x, mu):+.6f}   (symmetric)")
print(f"shift values and mu by 2 = {gmcc(x + 2, y + 2, mu + 2):+.6f}   (shift-invariant)")
print(f"scale values and mu by 3 = {gmcc(x * 3, y * 3, mu * 3):+.6f}   (scale-invariant)")
print(f"GMCC(x, x)               = {gmcc(x, x, mu):+.6f}   (self-similarity)")

GMCC(x, y)               = +0.130425
GMCC(y, x)               = +0.130425   (symmetric)
shift values and mu by 2 = +0.130425   (shift-invariant)
scale values and mu by 3 = +0.130425   (scale-invariant)
GMCC(x, x)               = +1.000000   (self-similarity)


**Equivalence with Pearson (Proposition 2).** When the centering references are aligned — that is, when $\mu$ coincides with the local mean used by Pearson — the two measures agree exactly. GMCC is therefore a reformulation of the *centering reference*, not of the correlation geometry itself.

In [13]:
a = rng.uniform(1, 5, size=10)
b = rng.uniform(1, 5, size=10)
b = b - b.mean() + a.mean()      # construct an aligned-centering pair
aligned_mu = float(a.mean())

print(f"Pearson(a, b)            = {pearson(a, b):+.10f}")
print(f"GMCC(a, b, aligned mu)   = {gmcc(a, b, aligned_mu):+.10f}")
print(f"difference               = {abs(pearson(a, b) - gmcc(a, b, aligned_mu)):.2e}")

Pearson(a, b)            = -0.2918979717
GMCC(a, b, aligned mu)   = -0.2918979717
difference               = 0.00e+00


## Summary

| | Centering reference | Estimated from | Baseline variance |
|---|---|---|---|
| Cosine | none (origin at 0) | — | — |
| Pearson | local mean $\bar{x}$, per entity | $n$ co-observations | $\sigma^2 / n$ |
| **GMCC** | **global mean $\mu$, shared** | **$N$ observations** | **$\sigma^2 / N$** |

Since $N \gg n$ under sparsity, the centering baseline is substantially more stable — a variance reduction by a factor of $n/N$ (Theorem 1). Per-pair cost stays $O(n)$, identical to Cosine and Pearson, with only $O(1)$ additional storage.

---

**Citation**

```bibtex
@article{wiyono2026gmcc,
  title   = {Global mean--centered correlation: A variance-reduced and
             theoretically grounded similarity measure for sparse data environments},
  author  = {Wiyono, Slamet and Adji, Teguh Bharata and
             Nugroho, Hanung Adi and Wibirama, Sunu},
  journal = {Information Sciences},
  volume  = {749},
  pages   = {123502},
  year    = {2026},
  doi     = {10.1016/j.ins.2026.123502}
}
```